In [1]:
%%writefile All_intents.py

import os
import re
import io
import email
import time
import json
import torch
import shutil
import imaplib
import smtplib
import tempfile
import pandas as pd
import fitz  
from PIL import Image
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from email.header import decode_header
from email.mime.text import MIMEText
from hijri_converter import Hijri
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever, ParentDocumentRetriever
from langchain.storage import InMemoryStore
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from googleapiclient.discovery import build
from google.oauth2 import service_account
from ArabicOcr import arabicocr  
import gc  
import openpyxl  
import json
import re
from datetime import datetime, timedelta
from hijri_converter import Hijri, Gregorian
from googleapiclient.discovery import build
from google.oauth2 import service_account
import json
import re
import re

from config import tokenizer, llm_pipeline, DEVICE, SERVICE_ACCOUNT_FILE, SCOPES
from RulesForIntents import RULES_DIR , load_intent_rules , deduce_administrative_rule , update_intent_rules_file  


Overwriting All_intents.py


# Calendar Handling 

In [2]:
%%writefile -a All_intents.py

try:
    credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
    service = build('calendar', 'v3', credentials=credentials)
except NameError:
    service = None 

def get_current_context():
    """استخراج السياق الزمني الحالي بدقة مع ضبط التوقيت للسعودية (UTC+3)"""
    now = datetime.utcnow() + timedelta(hours=3)
    
    days_map = {0: "الاثنين", 1: "الثلاثاء", 2: "الأربعاء", 3: "الخميس", 4: "الجمعة", 5: "السبت", 6: "الأحد"}
    day_name = days_map[now.weekday()]
    
    hijri_now = Gregorian(now.year, now.month, now.day).to_hijri()
    return f"اليوم هو {day_name}، التاريخ الميلادي: {now.strftime('%Y-%m-%d')}، التاريخ الهجري: {hijri_now.year}-{hijri_now.month}-{hijri_now.day}، الوقت الحالي: {now.strftime('%H:%M')}"

def arabic_to_english_numbers(s):
    if not s: return ""
    arabic_map = {'٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9'}
    return ''.join(arabic_map.get(char, char) for char in str(s))

def calculate_next_weekday(target_day_idx):
    """حساب تاريخ أقرب يوم أسبوع قادم رياضياً بناءً على الفهرس (0-6) بتوقيت السعودية"""
    if target_day_idx is None: return None

    today_ksa = datetime.utcnow() + timedelta(hours=3)
    current_idx = today_ksa.weekday()
    
    days_ahead = target_day_idx - current_idx
    
    if days_ahead <= 0:
        days_ahead += 7
        
    next_date = today_ksa + timedelta(days=days_ahead)
    return next_date.strftime("%Y-%m-%d")

def calculate_next_hijri(day, month, year=None):
    """حساب أقرب تاريخ ميلادي يوافق التاريخ الهجري المطلوب"""
    try:
        # نستخدم توقيت السعودية هنا أيضاً للاتساق
        now = datetime.utcnow() + timedelta(hours=3)
        current_hijri = Gregorian(now.year, now.month, now.day).to_hijri()
        
        if year is None:
            target_year = current_hijri.year
            # إذا كان الشهر واليوم قد مضى في هذه السنة الهجرية، ننتقل للسنة القادمة
            if (month < current_hijri.month) or (month == current_hijri.month and day < current_hijri.day):
                target_year += 1
        else:
            target_year = year
        
        greg_date = Hijri(target_year, month, day).to_gregorian()
        return greg_date.strftime("%Y-%m-%d")
    except Exception as e:
        print(f"Error calculating Hijri date: {e}")
        return None

def process_gregorian_date(date_str, extracted_data):
    """
    معالجة التاريخ الميلادي مع إعطاء الأولوية للتاريخ الصريح (يوم/شهر)
    على حساب حسابات أيام الأسبوع التقريبية.
    """
    if not date_str: return None
    clean_str = arabic_to_english_numbers(date_str).strip()

    day = extracted_data.get("day")
    month = extracted_data.get("month")
    year = extracted_data.get("year")
    
    now = datetime.utcnow() + timedelta(hours=3)
    
    if day and month:
        try:
            d = int(day)
            m = int(month)
            
            # تحديد السنة
            if year:
                y = int(year)
                # تصحيح السنوات المختصرة (مثال: 25 -> 2025)
                if y < 100: y += 2000
            else:
                # منطق إكمال السنة الناقصة
                y = now.year
                temp_dt = datetime(y, m, d)
                # إذا كان التاريخ في الماضي بالنسبة لليوم، نفترض السنة القادمة
                # مثال: نحن في ديسمبر والطلب "1 يناير"، إذن 1 يناير السنة القادمة
                if temp_dt.date() < now.date():
                    y += 1
            
            final_dt = datetime(y, m, d)
            return final_dt.strftime("%Y-%m-%d")
            
        except ValueError:
            # في حال كانت الأرقام غير منطقية، ننتقل للمحاولات الأخرى
            pass

    # 2. التعامل مع الكلمات الدلالية المباشرة "غدا" أو "اليوم"
    if "غدا" in clean_str.lower() or "غداً" in clean_str:
        return (now + timedelta(days=1)).strftime("%Y-%m-%d")
    if "اليوم" in clean_str.lower():
        return now.strftime("%Y-%m-%d")

    # 3. الملاذ الأخير: التعامل مع أيام الأسبوع (فقط إذا لم يذكر تاريخ رقمي)
    # مثال: "أريد موعد يوم الخميس القادم" (بدون ذكر تاريخ)
    target_day_idx = extracted_data.get("weekday_index")
    if target_day_idx is not None:
        return calculate_next_weekday(target_day_idx)
        
    return None

def process_hijri_date(date_str, extracted_data):
    """
    معالجة التاريخ الهجري بالاعتماد على تحليل LLM.
    """
    if not date_str: return None
    
    # الاعتماد المباشر على ما استخرجه الـ LLM
    day = extracted_data.get("day")
    month = extracted_data.get("month")
    year = extracted_data.get("year")
    
    if day and month:
        return calculate_next_hijri(int(day), int(month), int(year) if year else None)
    
    return None

def extract_appointment_info(email_body):
    """
    استخراج النية وبيانات الموعد باستخدام LLM، مع تفصيل اليوم/الشهر/السنة
    وتحويل أسماء الأيام والأشهر إلى أرقام باستخدام الـ prompt
    """
    current_time_context = get_current_context()
    
    system_prompt = f"""
أنت مساعد ذكي متخصص في استخراج بيانات المواعيد بدقة عالية. {current_time_context}.
مهمتك الرئيسية: تحليل النص بدقة لاستخراج النية والبيانات بصيغة JSON فقط.
كن قوياً ودقيقاً في الفهم، وتعامل مع الأخطاء الإملائية، التنسيقات المختلفة، والاختصارات.
افترض أن الطلب يتعلق بجدولة موعد جديد معك أو تأكيد موعد موجود إلا إذا كان واضحاً خلاف ذلك.

قواعد الاستخراج الأساسية:
- أعد النتيجة بصيغة JSON فقط بدون أي شرح أو نص إضافي.
- لا تُهمل أي تاريخ أو وقت مذكور صراحة في النص.
- إذا وُجد أكثر من تاريخ، اختر التاريخ المرتبط بالموعد أو الاجتماع.

قواعد تحديد النية (Intent Rules) - هام جداً:
- "schedule": 
  1. لطلب جدولة موعد جديد.
  2. إذا كان النص يقترح موعداً محدداً ويطلب التأكيد عليه (مثال: "هل نؤكد الموعد يوم كذا؟"، "أقترح يوم كذا"). في هذه الحالة، المستخدم يريد حجز هذا الوقت، لذا اعتبرها "schedule".
- "confirm": 
  1. فقط إذا كان السؤال يستفسر عن حالة موعد محجوز مسبقاً.
  2. لا تستخدم "confirm" إذا كان النص يحتوي على تاريخ ووقت جديدين يُراد تثبيتهما.

2. الفترات الزمنية المبهمة (Vague Times) - إلزامي:
   - إذا ذُكرت فترة بدون ساعة محددة، استخدم الأوقات الافتراضية التالية:
     - "صباحاً" أو "الصباح" أو "بداية الدوام" -> "09:00"
     - "فترة ما بعد الظهر" أو "الظهر" أو "بعد الظهر" -> "13:00"
     - "العصر" -> "16:00"
     - "المساء" أو "ليلاً" أو "في وقت متأخر" -> "19:00"
   - مثال: "يوم الثلاثاء فترة ما بعد الظهر" -> start_time: "13:00".
   
الحقول المطلوبة:
- "intent": 
  - "schedule" لطلب جدولة موعد جديد
  - "confirm" لتأكيد موعد
  - "none" إذا لم يكن النص متعلقاً بموعد
- "date": النص الخام للتاريخ كما ورد في الإيميل (مثال: "2 يناير القادم"، "2025/12/29"، "10 رجب 1447هـ").
- "date_type": 
  - "gregorian" للتاريخ الميلادي
  - "hijri" للتاريخ الهجري
- "day": رقم اليوم (1–31) إذا وجد.
- "month": رقم الشهر (1–12) إذا وجد.
- "year": رقم السنة إذا وجدت.
- "weekday_index": رقم يوم الأسبوع إذا ذُكر.
- "start_time": وقت بدء الموعد بصيغة 24 ساعة (HH:MM).
- "duration": مدة الاجتماع بالدقائق (افتراضي 60 إذا لم تُذكر).
- "title": عنوان مختصر وواضح للموعد (مثل: "اجتماع لمناقشة تعثر مؤسسة").

تحويل الأسماء إلى أرقام (إلزامي):
1) أيام الأسبوع:
- إذا ذُكر يوم أسبوع (مثل: الاثنين، الثلاثاء، الأربعاء، الخميس، الجمعة، السبت، الأحد)
  أعد "weekday_index" حسب الآتي:
  0 = الاثنين
  1 = الثلاثاء
  2 = الأربعاء
  3 = الخميس
  4 = الجمعة
  5 = السبت
  6 = الأحد

2) الأشهر الهجرية:
- إذا ذُكر شهر هجري (حتى مع أخطاء إملائية)، حوّله إلى رقم:
  1 محرم
  2 صفر
  3 ربيع الأول
  4 ربيع الثاني
  5 جمادى الأولى
  6 جمادى الثانية
  7 رجب
  8 شعبان
  9 رمضان
  10 شوال
  11 ذو القعدة
  12 ذو الحجة
- تعامل مع صيغ مثل: "جمادي"، "ذو القعده"، "ربيع الاول".

3) الأشهر الميلادية العربية:
- إذا ذُكر شهر ميلادي عربي، حوّله إلى "month" كرقم:
  1 يناير
  2 فبراير
  3 مارس
  4 أبريل
  5 مايو
  6 يونيو
  7 يوليو
  8 أغسطس
  9 سبتمبر
  10 أكتوبر
  11 نوفمبر
  12 ديسمبر
- تعامل مع صيغ مثل: "اكتوبر"، "ديسيمبر"، "جانفي".

قواعد استكمال التاريخ الناقص:
- إذا ذُكر اليوم والشهر فقط (مثل: "2 يناير"):
  - استخدم السنة الحالية، وإذا كان التاريخ قد مضى فاجعل السنة القادمة.
- إذا ذُكر "القادم":
  - اختر أقرب تاريخ قادم من السياق الزمني الحالي.
- إذا ذُكر يوم أسبوع فقط:
  - أعد "weekday_index" ولا تُخمن اليوم الرقمي.
- إذا ذُكر تاريخ هجري بدون سنة:
  - استخدم السنة الهجرية الحالية أو القادمة حسب السياق.

قواعد الوقت:
- إذا ذُكر "الساعة السادسة مساء":
  - حوّلها إلى "18:00".
- إذا لم يُذكر وقت:
  - استخدم "09:00" افتراضياً.

قواعد التعامل مع التواريخ والأوقات النسبية (مثل "اليوم"، "غداً"، "بعد 10 دقائق"، "بعد ساعة"):
- استخدم السياق الزمني الحالي لحساب التاريخ والوقت المطلق.
- أعد "day"، "month"، "year"، "start_time" كقيم مطلقة محسوبة.
- مثال: إذا كان "بعد 10 دقائق" والوقت الحالي 14:00، فـ "start_time": "14:10"، و"day"/"month"/"year" الحالي.
- مثال: إذا كان "غداً الساعة 9 صباحاً"، فـ "day"/"month"/"year" للغد، "start_time": "09:00".
- لـ "بعد ساعة"، أضف ساعة واحدة إلى الوقت الحالي وحدث التاريخ إذا لزم.
- تأكد من أن النتيجة دائماً في المستقبل أو الحاضر إذا أمكن.

قواعد صارمة لتحويل الوقت (إلزامي بدون استثناء):
- أي وقت يُذكر مع كلمات تدل على المساء يجب تحويله إلى نظام 24 ساعة بإضافة 12 ساعة إذا كان بين 1 و 11.
- كلمات المساء تشمل (على سبيل المثال لا الحصر):
  مساء، مساً، م، PM، بعد الظهر، ظهراً، الظهر، العصر، عصراً، المغرب، ليلاً، الليل.
- أمثلة إلزامية:
  - "1:00 ظهراً" → "13:00"
  - "الساعة 3 العصر" → "15:00"
  - "8 مساءً" → "20:00"
  - "11 ليلاً" → "23:00"
- أي وقت يُذكر مع كلمات تدل على الصباح يُبقى كما هو (1–11).
- كلمات الصباح تشمل:
  صباحاً، صباح، ص، AM، الفجر، فجراً، الضحى.
- أمثلة:
  - "1:00 صباحاً" → "01:00"
  - "9 صباح" → "09:00"
- إذا ذُكرت ساعة فقط بدون دقائق (مثل: "الساعة 1 ظهراً"):
  - افترض الدقائق "00".
  - مثال: "الساعة 1 ظهراً" → "13:00"
- إذا ذُكر وقت رقمي مع كلمة مساء أو ظهراً (مثل "1:00 م"):
  - اعتبره وقتاً مسائياً دائماً.

أعد النتيجة النهائية بصيغة JSON فقط، بدون أي تعليق أو نص إضافي.
"""
    
    prompt = f"حلل هذا النص بدقة: {email_body}"
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]
    
    # محاكاة استدعاء LLM (يفترض وجود tokenizer و llm_pipeline لـ Qwen 32b)
    # تأكد أن المتغيرات tokenizer و llm_pipeline معرفة في النطاق العام
    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        raw_output = llm_pipeline(text_input)[0]['generated_text'].replace(text_input, "").strip()
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except Exception as e:
        print(f"LLM Error: {e}")
        return {"intent": "none"}
    return {"intent": "none"}

def generate_llm_response(context_data):
    system_prompt =   """
أنت مدير إدارة المشاريع. اكتب رداً رسمياً سعودياً بليغاً باللغة العربية الفصحى.

قواعد الهوية (صارمة جداً):
1. استخدم اسم المرسل فقط إذا كان موجوداً صراحة في السياق.
2. إذا لم يوجد اسم مرسل واضح، استخدم صيغة عامة مثل:
   - "سعادة الإخوة الكرام"
   - "سعادة ممثل الجهة"
3. يُمنع منعاً باتاً اختراع أي اسم أو نسب أو كنية.
4. إذا وُجد منصب وظيفي، استخدمه بعد الاسم.
5. لا تستخدم أمثلة توضيحية أو أقواس أو عبارات مثل (إن وجد).

قواعد الصياغة:
- النبرة: رسمية، سعودية، مهنية، مباشرة.
- لا تغيّر تاريخ أو وقت أو مدة الموعد.
- لا تضف أي معلومات غير موجودة في السياق.

التوقيع الإلزامي في نهاية الإيميل:
وتقبلوا فائق التقدير والاحترام،،
مدير إدارة المشاريع
"""
    user_prompt = f"سياق الرد: {context_data}. اكتب الرد النهائي."
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    
    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        return llm_pipeline(text_input)[0]['generated_text'].replace(text_input, "").strip()
    except:
        return "تمت المعالجة."

def handle_calendar_intent(email_body, sender_email):
    """
    المعالج الرئيسي: استخراج النية، حفظ البيانات في JSON، معالجة التاريخ (مع التعامل مع الناقص)، وإدارة التقويم
    """
    
    print(f"\n--- [New Email Processing] Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ---")
    
    info = extract_appointment_info(email_body)
    intent = info.get("intent", "none")
    
    if intent == "none" or not info.get("date"):
        return None
    
    # 2. حفظ بيانات الموعد في ملف JSON
    json_filename = f"appointment_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(json_filename, 'w', encoding='utf-8') as json_file:
        json.dump(info, json_file, ensure_ascii=False, indent=4)
    print(f"تم حفظ بيانات الموعد في: {json_filename}")
    
    target_date = None
    extracted_data = {
        "day": info.get("day"),
        "month": info.get("month"),
        "year": info.get("year"),
        "weekday_index": info.get("weekday_index")
    }
    if info.get("date_type") == "hijri":
        target_date = process_hijri_date(info["date"], extracted_data)
    else:
        target_date = process_gregorian_date(info["date"], extracted_data)
    
    if not target_date:
        return generate_llm_response({"status": "error_date", "original_date": info["date"]})
    
    start_time = arabic_to_english_numbers(info.get("start_time", "09:00"))
    if ":" not in start_time: start_time = f"{start_time.zfill(2)}:00"
    
    try:
        start_dt = datetime.strptime(f"{target_date} {start_time}", "%Y-%m-%d %H:%M")
        start_iso = start_dt.isoformat() + "+03:00"
        end_iso = (start_dt + timedelta(minutes=info.get("duration", 60))).isoformat() + "+03:00"
    except:
        return generate_llm_response({"status": "error_time"})
    
    if intent == "schedule":
        # التأكد من عدم وجود متغير service كـ None
        if service:
            check_body = {"timeMin": start_iso, "timeMax": end_iso, "items": [{"id": CALENDAR_ID}]}
            busy_slots = service.freebusy().query(body=check_body).execute()['calendars'][CALENDAR_ID]['busy']
            
            if not busy_slots:
                event = {
                    'summary': info.get("title", "اجتماع"),
                    'description': f"جدولة آلية من: {sender_email}",
                    'start': {'dateTime': start_iso, 'timeZone': 'Asia/Riyadh'},
                    'end': {'dateTime': end_iso, 'timeZone': 'Asia/Riyadh'},
                }
                service.events().insert(calendarId=CALENDAR_ID, body=event).execute()
                return generate_llm_response({"status": "booked", "date": target_date, "time": start_time, "title": info["title"]})
            else:
                return generate_llm_response({"status": "busy", "date": target_date, "time": start_time})
        else:
            return "عذراً، خدمة التقويم غير متصلة."
            
    elif intent == "confirm":
        if service:
            events_result = service.events().list(
                calendarId=CALENDAR_ID, timeMin=target_date+"T00:00:00+03:00",
                timeMax=target_date+"T23:59:59+03:00", singleEvents=True
            ).execute()
            events = events_result.get('items', [])
            
            if events:
                return generate_llm_response({"status": "confirmed_exists", "date": target_date, "event_title": events[0]['summary']})
            else:
                return generate_llm_response({"status": "not_found", "date": target_date})
        else:
            return "عذراً، خدمة التقويم غير متصلة."
            
    return None

Appending to All_intents.py


# Handling genral inquiries 

In [3]:
%%writefile -a All_intents.py


def get_isolated_knowledge_context(query, kb):
    """
    استرجاع السياق من ملفات محددة فقط لضمان عزلة الرد على الاستفسارات.
    الملفات المستهدفة: الأنظمة، المستودعات، الشروط الخاصة، وسجل الإيميلات.
    """
    allowed_files = [
        "modified_comp_updated3 (1).txt", 
        "Depot_Updated.txt", 
        "private_cond.txt", 
        "resident_cleaned.txt", 
        "email_history.txt"
    ]
    
    # 1. استرجاع أولي مكثف
    faiss_docs = kb["vs_large"].as_retriever(search_kwargs={"k": 10}).invoke(query)
    bm25_docs = kb["bm25_large"].invoke(query)
    
    all_docs = faiss_docs + bm25_docs
    
    unique_content = set()
    filtered_context = []
    
    for doc in all_docs:
        # التحقق من أن مصدر القطعة المسترجعة ضمن الملفات المسموح بها فقط
        source_name = os.path.basename(doc.metadata.get('source', ''))
        if source_name in allowed_files:
            if doc.page_content not in unique_content:
                filtered_context.append(doc.page_content)
                unique_content.add(doc.page_content)
                
    return "\n---\n".join(filtered_context)

Appending to All_intents.py


# Handling genral inquiries 

In [4]:
%%writefile -a All_intents.py

def handle_general_inquiries(email_body, sender_info, kb):
    """
    تحليل الإيميل ومعرفة ما إذا كان استفساراً عاماً والإجابة عليه من الأنظمة المحددة فقط.
    """
    # استرجاع السياق المعزول أولاً
    isolated_context = get_isolated_knowledge_context(email_body, kb)
    
    if not isolated_context.strip():
        return None

    # البرومبت المطور لضمان الالتزام بالمخرجات فقط
    system_prompt = """
أنت خبير متخصص في صياغة الخطابات الرسمية باللغة العربية الفصحى. مهمتك هي تحليل البريد الإلكتروني الوارد والرد عليه وفق القواعد الصارمة التالية:

1. قاعدة التصنيف (هام جداً):
   - حدد ما إذا كان الإيميل "استفساراً عاماً" (طلب معلومات، بيانات موظف، إجراء نظامي).
   - إذا كان الإيميل يتعلق بـ (صيانة، أعطال، إصلاح فني، مشكلة تقنية)، اعتبره فوراً "غير استفسار".
   - إذا لم يكن استفساراً عاماً، رُد حصراً بكلمة: NOT_GENERAL_INQUIRY (بدون أي زيادة).

2. قاعدة المخرجات (شكل الرد):
   - في حال كان الإيميل استفساراً عاماً، يجب أن تقتصر مخرجاتك "فقط وحصراً" على نص الرد الرسمي.
   - يمنع منعاً باتاً كتابة أي مقدمات مثل "التصنيف:" أو "بناءً على التحليل" أو أي شرح لسبب الإجابة. ابدأ مباشرة بالتحية الرسمية.

3. بروتوكول الصياغة الرسمي:
   - اللغة: العربية الفصحى الرصينة.
   - المخاطبة: 
     * إذا كان المرسل "مدير"، ابدأ بـ: "سعادة مدير [اسم الإدارة]، السلام عليكم ورحمة الله وبركاته،".
     * إذا لم يكن مديراً، استخدم منصبه الوارد أو "السيد/ [الاسم]".
   - المتن: أجب بناءً على "السياق المرفق" فقط. لا تضف معلومات خارجية.
   - الخاتمة: التزم بالخاتمة التالية نصاً: "وتقبلوا تحياتي، مدير قسم الإشراف."

4. محذورات:
   - لا تذكر للطرف الآخر أنك استخرجت المعلومات من "سياق مرفق".
   - لا تخرج عن نطاق المعلومات المتوفرة.
"""
    
    prompt = f"السياق المتاح للرد:\n{isolated_context}\n\nنص الإيميل الوارد:\n{email_body}\nمعلومات المرسل:\n{sender_info}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    try:
        # استخدام apply_chat_template
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True  , enable_thinking=False)
        
        # إعدادات التوليد لضمان أقل قدر من الهلوسة
        output = llm_pipeline(
            text_input, 
            temperature=0.01, # خفض درجة الحرارة لأقصى حد لزيادة الدقة
            max_new_tokens=1024,
            do_sample=False # لضمان إجابة محددة Deterministic
        )[0]['generated_text'].replace(text_input, "").strip()
        
        # التحقق من شرط عدم الاختصاص
        if "NOT_GENERAL_INQUIRY" in output:
            return None
            
        return output
    except Exception as e:
        print(f"⚠ خطأ في معالجة الاستفسار العام: {e}")
        return None


Appending to All_intents.py


# handle administrative procedures

In [5]:
%%writefile -a All_intents.py
def handle_administrative_procedures(email_body, sender_info):
    """
    نسخة نصية مباشرة (No-JSON): تعتمد على صياغة الرد فوراً بناءً على القواعد.
    """
    # تحميل القواعد المكتسبة سابقاً
    learned_rules = load_intent_rules("ADMIN")
    
    system_prompt = f"""
أنت مدير قسم الإشراف (شخصية إدارية حازمة). مهمتك صياغة الخطابات الرسمية فقط.

التعليمات الصارمة:
1. اكتب نص الرد الرسمي مباشرة.
2. يمنع كتابة أي مقدمات مثل "بناءً على طلبك.." أو "إليك الرد..".
3. يمنع استخدام صيغة JSON أو أي أكواد برمجية.
4. التزم بالقواعد الإدارية التالية المستنتجة من المدير:
{learned_rules if learned_rules else "اتبع السياق العام للردود الإدارية."}

الصيغة المطلوبة:
- ابدأ بالتحية الرسمية فوراً.
- ادخل في صلب الموضوع.
- اختم وجوباً بعبارة: "وتقبلوا تحياتي، مدير قسم الإشراف".
"""

    user_input = f"المرسل: {sender_info}\nنص المعاملة الواردة:\n{email_body}\n\nاكتب الرد الرسمي الآن:"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    try:
        # استخدام التمبلت لتجهيز النص
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        
        # توليد الرد (درجة حرارة منخفضة لضمان الالتزام)
        outputs = llm_pipeline(text_input, max_new_tokens=1000, temperature=0.01)
        
        # استخراج النص وتنظيفه من المدخلات
        response_text = outputs[0]['generated_text'].replace(text_input, "").strip()
        
        # تنظيف إضافي (احتياطي) في حال وضع النموذج علامات اقتباس
        response_text = response_text.replace('```', '').replace('json', '').strip()
        
        return response_text
                
    except Exception as e:
        print(f"⚠ خطأ في المعالجة الإدارية (Text-Mode): {e}")
        return None

Appending to All_intents.py


# handle HR procedures

In [6]:
%%writefile -a All_intents.py
def handle_hr_procedures(email_body, sender_info):
    """
    نسخة نصية مباشرة (No-JSON): للموارد البشرية.
    تم تحديثها لتعطي أولوية قصوى للقواعد المتعلمة.
    """
    # تحميل القواعد المكتسبة
    learned_rules = load_intent_rules("HR")
    
    # طباعة للتحقق من أن القواعد تم تحميلها فعلاً (Debug)
    if learned_rules:
        print(f"✅ [HR] تم تحميل قواعد متعلمة: {learned_rules[:5000]}...")
    else:
        print("ℹ️ [HR] لا توجد قواعد متعلمة، سيتم استخدام القواعد الافتراضية.")

    system_prompt = f"""
أنت مدير قسم الإشراف المتمرس. مهمتك كتابة الرد الرسمي النهائي.

المرجع التشغيلي (Operating Manual) الخاص بك هو القواعد التالية، ويجب تطبيقها بصرامة:
--------------------------------------------------
{learned_rules if learned_rules else "طبق القواعد الإدارية العامة المتعارفت عليها في السعودية."}
--------------------------------------------------

تعليمات التنفيذ:
1. تحقق من القواعد أعلاه: هل توجد قاعدة خاصة بهذا النوع من الطلبات؟ (مثال: هل يجب توجيه الرد لطرف ثالث؟).
2. استخدم أي "أرقام مواد" أو "بنود تعاقدية" مذكورة في القواعد إذا انطبقت الحالة.
3. التزم بصيغة: "سعادة / [المنصب المناسب حسب القواعد]... السلام عليكم ورحمة الله... [المتن]... وتقبلوا تحياتي" مدير قسم الإشراف
4. لا تكتب أي شرح، فقط نص الرسالة.
"""

    prompt = f"المرسل: {sender_info}\nالطلب: {email_body}\n\nصغ الرد الرسمي مباشرة بناءً على التعليمات أعلاه:"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        
        # رفعنا التوكنز قليلاً لأن الرد قد يكون مفصلاً
        outputs = llm_pipeline(text_input, max_new_tokens=1024, temperature=0.01)
        
        response_text = outputs[0]['generated_text'].replace(text_input, "").strip()
        
        # تنظيف أي شوائب قد يضيفها النموذج
        response_text = response_text.replace('```', '').strip()
        
        return response_text

    except Exception as e:
        print(f"⚠ خطأ في معالجة الموارد البشرية (Text-Mode): {e}")
        return None

Appending to All_intents.py


# handle MAINTENANCE procedures

In [7]:
%%writefile -a All_intents.py
def rag_answer_email(user_query, context):
    """
    نسخة محدثة: تدمج القواعد المتعلمة للنية العامة (RAG_MAINTENANCE).
    """
    # تحميل القواعد المكتسبة للصيانة
    learned_rules = load_intent_rules("RAG_MAINTENANCE")
    
    system_prompt = f"""
أنت موظف إداري متخصص في كتابة الإيميلات الرسمية الحكومية، وتتحدث بصفتك "مدير قسم الإشراف".
مهمتك: إدارة بلاغات الصيانة والردود الرسمية بأسلوب مهني حازم.

--- تعليمات المدير المحدثة (Learned Rules) ---
{learned_rules if learned_rules else "لا توجد تعليمات محدثة، طبق القواعد العامة."}
----------------------------------------------

القواعد العامة:
1. الالتزام باللغة العربية الفصحى الرسمية المتبعة في الخطابات الحكومية.
2. عدم اختراع أو افتراض أي معلومات غير موجودة في سياق البيانات المتاحة.
3. يجب أن تنتهي الرسالة دائماً وبشكل نصي بعبارة: 
وتقبلو تحياتي مدير قسم الاشراف
"""

    prompt = f"{system_prompt}\n\n--- السياق المتاح ---\n{context}\n\n--- رسالة البريد الإلكتروني الواردة ---\n{user_query}\n\nالرد:"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    outputs = llm_pipeline(text_input)
    response_text = outputs[0]['generated_text'].replace(text_input, "").strip()
    
    if response_text.startswith("الرد:"): response_text = response_text[len("الرد:"):].strip()
    return response_text

Appending to All_intents.py


# handle attachment procedures

In [8]:
%%writefile -a All_intents.py
def handle_attachment_special_requests(attachments, query, processor, embed_model, sender_info):
    """
    دالة متخصصة لمعالجة المرفقات فقط واتخاذ إجراءات بناءً على محتواها التقني
    مثل توجيه الطلبات لشركة المياه الوطنية.
    """
    if not attachments:
        return None

    # 1. استخراج السياق من المرفقات فقط باستخدام نظام Small-to-Parent
    attachment_ctx = get_attachment_context(attachments, query, processor, embed_model)
    
    if not attachment_ctx or "###" not in attachment_ctx:
        return None

    # 2. برومبت فحص محتوى المرفقات واتخاذ القرار
    system_prompt = """
    أنت خبير إداري وفني في قسم الإشراف. مهمتك هي تحليل "سياق المرفقات" المرفق فقط.
    
    القاعدة الصارمة:
    1. ابحث في المرفقات عن أي إشارة لمواقع، عقارات، أو مخططات ذكر فيها أنها "غير موصولة بشبكة المياه"، "تفتقر للخدمة"، "تحتاج إيصال مياه"، أو "خارج نطاق التغطية الحالية".
    2. إذا وجدت ذلك: قم بصياغة خطاب رسمي موجه إلى (السادة/ شركة المياه الوطنية) تطلب فيه "إيصال خدمة المياه للمواقع المذكورة" بناءً على البيانات المستخرجة من المرفقات.
    3. يجب أن يتضمن الخطاب: أسماء المواقع أو أرقام القطع إن وجدت، والتأكيد على أهمية الخدمة.
    4. الخاتمة ثابتة: "وتقبلوا تحياتي، مدير قسم الإشراف".
    5. إذا لم تجد أي إشارة لمواقع تحتاج مياه في المرفقات، رُد حصراً بكلمة: NO_WATER_ACTION_REQUIRED
    
    الصياغة يجب أن تكون رسمية جداً وباللغة العربية الفصحى.
    """

    user_input = f"سياق المرفقات المستخرج:\n{attachment_ctx}\n\nبيانات المرسل:\n{sender_info}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        output = llm_pipeline(
            text_input, 
            temperature=0.1, 
            max_new_tokens=1024,
            do_sample=True
        )[0]['generated_text'].replace(text_input, "").strip()

        if "NO_WATER_ACTION_REQUIRED" in output:
            return None
        
        return output
    except Exception as e:
        print(f"⚠ خطأ في معالجة طلبات المرفقات الخاصة: {e}")
        return None

Appending to All_intents.py


# handle User procedures

In [9]:
%%writefile -a All_intents.py
def handle_User_Requests(email_body, sender_info, kb, attachments, processor, embed_model):

    """
    دالة مرنة تتيح للمطور تحديد مصادر المعلومات والمرفقات بسهولة عبر التعليقات (#).
    """
    
    # ---------------------------------------------------------
    # 1. منطقة الإعدادات (Control Panel)
    # أضف # أمام السطر لتعطيل المصدر، أو احذفه لتفعيله
    # ---------------------------------------------------------
    
    # قائمة الملفات التي تريد البحث فيها لهذا النوع من الإيميلات حصراً
    allowed_knowledge_files = [
        #"modified_comp_updated3 (1).txt",  # ملف العقود
         "Depot_Updated.txt",             # <-- تم تعطيله بوضع #
        #"private_cond.txt",                # الشروط الخاصة
        # "resident_cleaned.txt",          # <-- تم تعطيله
        "email_history.txt"
    ]

    # هل تريد السماح بقراءة المرفقات الحالية في الإيميل؟
    USE_ATTACHMENTS = True   # اجعلها False لتجاهل المرفقات في هذا الرد
    
    # هل تريد استخدام سجل الإيميلات السابق (الذاكرة)؟
    USE_EMAIL_HISTORY = True 
    
    # اسم النية أو الشخصية التي سيتحدث بها البوت
    ROLE_NAME = "مدير المشاريع الخاصة"
    
    # ---------------------------------------------------------
    # 2. بناء السياق بناءً على الإعدادات أعلاه
    # ---------------------------------------------------------
    final_context = []

    # أ) استرجاع المعلومات من الملفات المحددة فقط (Filtered RAG)
    if allowed_knowledge_files:
        # استرجاع أولي واسع
        faiss_docs = kb["vs_large"].as_retriever(search_kwargs={"k": 15}).invoke(email_body)
        bm25_docs = kb["bm25_large"].invoke(email_body)
        all_docs = faiss_docs + bm25_docs
        
        unique_content = set()
        for doc in all_docs:
            source_name = os.path.basename(doc.metadata.get('source', ''))
            # الشرط الجوهري: هل الملف موجود في القائمة المسموحة؟
            if source_name in allowed_knowledge_files:
                if doc.page_content not in unique_content:
                    final_context.append(f"[{source_name}]: {doc.page_content}")
                    unique_content.add(doc.page_content)

    # ب) إضافة المرفقات إذا كانت مفعلة
    if USE_ATTACHMENTS and attachments:
        att_context = get_attachment_context(attachments, email_body, processor, embed_model)
        if att_context:
            final_context.append(f"--- محتوى المرفقات الحالية ---\n{att_context}")

    # ج) إضافة سجل الإيميلات إذا كان مفعلاً
    # ج) إضافة سجل الإيميلات (تعديل لقراءة آخر الأحداث زمنياً)
# ج) إضافة سجل الإيميلات إذا كان مفعلاً (نسخة مطابقة لآلية Inquiry)
    if USE_EMAIL_HISTORY:
        # 1. استخدام البحث الهجين (Hybrid Retrieval) لضمان الدقة والشمولية
        
        # أ) بحث دلالي (Vector Search) - رفعنا العدد لـ 10
        faiss_history = kb["vs_large"].as_retriever(search_kwargs={"k": 10}).invoke(email_body)
        
        # ب) بحث بالكلمات المفتاحية (Keyword Search) - لاسترجاع الأسماء والأرقام بدقة
        bm25_history = kb["bm25_large"].invoke(email_body)
        
        # دمج النتائج
        all_history_hits = faiss_history + bm25_history
        
        # 2. التصفية ومنع التكرار (Filtering & Deduplication)
        history_unique = set()
        
        for doc in all_history_hits:
            # التحقق الصارم أن المصدر هو ملف السجل فقط
            if "email_history.txt" in doc.metadata.get('source', ''):
                # منع تكرار نفس النص إذا ظهر في البحثين
                if doc.page_content not in history_unique:
                    final_context.append(f"--- من الذاكرة السابقة (Hybrid) ---\n{doc.page_content}")
                    history_unique.add(doc.page_content)

    # تحويل القائمة إلى نص
    context_str = "\n\n".join(final_context)
    
    if not context_str.strip():
        # إذا لم نجد أي معلومة في المصادر المحددة، هل تريد رداً عاماً أم تجاهل؟
        # هنا سنقوم بإرجاع None ليقوم البوت بالرد العام أو تجاهله
        return None

    # ---------------------------------------------------------
    # 3. استدعاء النموذج (LLM)
    # ---------------------------------------------------------
    system_prompt = f"""
أنت موظف متخصص في كتابة الايميلات الرسمية. مهمتك الرد على البريد بمهنية عالية بناءً على السياق المقدم فقط.

القواعد:
1. استخدم المعلومات الواردة في "السياق المتاح" بدقة.
2. إذا كانت المعلومات ناقصة، اطلب التوضيح ولا تختلق معلومات.
3. التزم باللغة العربية الفصحى الرسمية.
4. التوقيع: {ROLE_NAME}.
5. لاتتخيل او تخترع اي معلومات او فقرات نظامية غير موجودة
"""

    user_prompt = f"""
معلومات المرسل: {sender_info}
نص الرسالة: {email_body}

--- السياق المتاح (المصادر والمرفقات) ---
{context_str}

الرد المقترح:
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        
        # إعدادات التوليد (يمكنك تعديل الحرارة هنا)
        outputs = llm_pipeline(
            text_input, 
            temperature=0.15, 
            max_new_tokens=1024,
            do_sample=True
        )
        return outputs[0]['generated_text'].replace(text_input, "").strip()
        
    except Exception as e:
        print(f"⚠ خطأ في الدالة المرنة: {e}")
        return None

Appending to All_intents.py


In [10]:
%%writefile -a All_intents.py
def handle_administrative_procedures_2(email_body, sender_info, kb, attachments, processor, embed_model):

    """
    دالة مرنة تتيح للمطور تحديد مصادر المعلومات والمرفقات بسهولة عبر التعليقات (#).
    """
    
    # ---------------------------------------------------------
    # 1. منطقة الإعدادات (Control Panel)
    # أضف # أمام السطر لتعطيل المصدر، أو احذفه لتفعيله
    # ---------------------------------------------------------
    
    # قائمة الملفات التي تريد البحث فيها لهذا النوع من الإيميلات حصراً
    allowed_knowledge_files = [
        #"modified_comp_updated3 (1).txt",  # ملف العقود
         #"Depot_Updated.txt",             # <-- تم تعطيله بوضع #
        #"private_cond.txt",                # الشروط الخاصة
        # "resident_cleaned.txt",          # <-- تم تعطيله
        #"email_history.txt"
    ]

    # هل تريد السماح بقراءة المرفقات الحالية في الإيميل؟
    USE_ATTACHMENTS = True   # اجعلها False لتجاهل المرفقات في هذا الرد
    
    # هل تريد استخدام سجل الإيميلات السابق (الذاكرة)؟
    USE_EMAIL_HISTORY = False 
    
    # اسم النية أو الشخصية التي سيتحدث بها البوت
    ROLE_NAME = "مدير المشاريع الخاصة"
    
    # ---------------------------------------------------------
    # 2. بناء السياق بناءً على الإعدادات أعلاه
    # ---------------------------------------------------------
    final_context = []

    # أ) استرجاع المعلومات من الملفات المحددة فقط (Filtered RAG)
    if allowed_knowledge_files:
        # استرجاع أولي واسع
        faiss_docs = kb["vs_large"].as_retriever(search_kwargs={"k": 15}).invoke(email_body)
        bm25_docs = kb["bm25_large"].invoke(email_body)
        all_docs = faiss_docs + bm25_docs
        
        unique_content = set()
        for doc in all_docs:
            source_name = os.path.basename(doc.metadata.get('source', ''))
            # الشرط الجوهري: هل الملف موجود في القائمة المسموحة؟
            if source_name in allowed_knowledge_files:
                if doc.page_content not in unique_content:
                    final_context.append(f"[{source_name}]: {doc.page_content}")
                    unique_content.add(doc.page_content)

    # ب) إضافة المرفقات إذا كانت مفعلة
    if USE_ATTACHMENTS and attachments:
        att_context = get_attachment_context(attachments, email_body, processor, embed_model)
        if att_context:
            final_context.append(f"--- محتوى المرفقات الحالية ---\n{att_context}")

    # ج) إضافة سجل الإيميلات إذا كان مفعلاً
    # ج) إضافة سجل الإيميلات (تعديل لقراءة آخر الأحداث زمنياً)
# ج) إضافة سجل الإيميلات إذا كان مفعلاً (نسخة مطابقة لآلية Inquiry)
    if USE_EMAIL_HISTORY:
        # 1. استخدام البحث الهجين (Hybrid Retrieval) لضمان الدقة والشمولية
        
        # أ) بحث دلالي (Vector Search) - رفعنا العدد لـ 10
        faiss_history = kb["vs_large"].as_retriever(search_kwargs={"k": 10}).invoke(email_body)
        
        # ب) بحث بالكلمات المفتاحية (Keyword Search) - لاسترجاع الأسماء والأرقام بدقة
        bm25_history = kb["bm25_large"].invoke(email_body)
        
        # دمج النتائج
        all_history_hits = faiss_history + bm25_history
        
        # 2. التصفية ومنع التكرار (Filtering & Deduplication)
        history_unique = set()
        
        for doc in all_history_hits:
            # التحقق الصارم أن المصدر هو ملف السجل فقط
            if "email_history.txt" in doc.metadata.get('source', ''):
                # منع تكرار نفس النص إذا ظهر في البحثين
                if doc.page_content not in history_unique:
                    final_context.append(f"--- من الذاكرة السابقة (Hybrid) ---\n{doc.page_content}")
                    history_unique.add(doc.page_content)

    # تحويل القائمة إلى نص
    context_str = "\n\n".join(final_context)
    
    if not context_str.strip():
        # إذا لم نجد أي معلومة في المصادر المحددة، هل تريد رداً عاماً أم تجاهل؟
        # هنا سنقوم بإرجاع None ليقوم البوت بالرد العام أو تجاهله
        return None

    # ---------------------------------------------------------
    # 3. استدعاء النموذج (LLM)
    # ---------------------------------------------------------
    system_prompt = f"""
أنت موظف متخصص في كتابة الايميلات الرسمية. مهمتك الرد على البريد بمهنية عالية بناءً على السياق المقدم فقط.

القواعد:
1. استخدم المعلومات الواردة في "السياق المتاح" بدقة.
2. إذا كانت المعلومات ناقصة، اطلب التوضيح ولا تختلق معلومات.
3. التزم باللغة العربية الفصحى الرسمية.
4. التوقيع: {ROLE_NAME}.
5. لاتتخيل او تخترع اي معلومات او فقرات نظامية غير موجودة
"""

    user_prompt = f"""
معلومات المرسل: {sender_info}
نص الرسالة: {email_body}

--- السياق المتاح (المصادر والمرفقات) ---
{context_str}

الرد المقترح:
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    try:
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        
        # إعدادات التوليد (يمكنك تعديل الحرارة هنا)
        outputs = llm_pipeline(
            text_input, 
            temperature=0.15, 
            max_new_tokens=1024,
            do_sample=True
        )
        return outputs[0]['generated_text'].replace(text_input, "").strip()
        
    except Exception as e:
        print(f"⚠ خطأ في الدالة المرنة: {e}")
        return None

Appending to All_intents.py
